In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName("MyApp")\
.master("local[*]")\
.enableHiveSupport()\
.getOrCreate()
sc = spark.sparkContext 

spark
spark.sparkContext

<SparkContext master=local[*] appName=MyApp>

In [9]:
myCollection = "My Name Is Diva and I love Spark".split(" ")

In [10]:
words = spark.sparkContext.parallelize(myCollection, 2)

In [11]:
words.map(lambda word: (word.lower(), 1))

PythonRDD[4] at RDD at PythonRDD.scala:53

In [12]:
words.map(lambda word: (word.lower(), 1)).collect()

[('my', 1),
 ('name', 1),
 ('is', 1),
 ('diva', 1),
 ('and', 1),
 ('i', 1),
 ('love', 1),
 ('spark', 1)]

In [13]:
keyword = words.keyBy(lambda word: word.lower()[0])

In [14]:
# mapping over key-values
keyword.mapValues(lambda word: word.upper()).collect()

[('m', 'MY'),
 ('n', 'NAME'),
 ('i', 'IS'),
 ('d', 'DIVA'),
 ('a', 'AND'),
 ('i', 'I'),
 ('l', 'LOVE'),
 ('s', 'SPARK')]

In [16]:
keyword.keys().collect()

['m', 'n', 'i', 'd', 'a', 'i', 'l', 's']

In [17]:
keyword.values().collect()

['My', 'Name', 'Is', 'Diva', 'and', 'I', 'love', 'Spark']

In [18]:
# to find specific values
keyword.lookup("d")

['Diva']

In [20]:
keyword.lookup("i")

['Is', 'I']

In [21]:
chars = words.flatMap(lambda word: word.lower())
KVcharacters = chars.map(lambda letter:(letter,1))

In [24]:
chars.collect()
KVcharacters.collect()

[('m', 1),
 ('y', 1),
 ('n', 1),
 ('a', 1),
 ('m', 1),
 ('e', 1),
 ('i', 1),
 ('s', 1),
 ('d', 1),
 ('i', 1),
 ('v', 1),
 ('a', 1),
 ('a', 1),
 ('n', 1),
 ('d', 1),
 ('i', 1),
 ('l', 1),
 ('o', 1),
 ('v', 1),
 ('e', 1),
 ('s', 1),
 ('p', 1),
 ('a', 1),
 ('r', 1),
 ('k', 1)]

In [25]:
def maxFunc(left, right):
    return max(left, right)

def addFunc(left, right):
    return left + right

nums = sc.parallelize(range(1,31),5)

In [26]:
nums.collect()

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30]

In [27]:
KVcharacters.countByKey()

defaultdict(int,
            {'m': 2,
             'y': 1,
             'n': 2,
             'a': 4,
             'e': 2,
             'i': 3,
             's': 2,
             'd': 2,
             'v': 2,
             'l': 1,
             'o': 1,
             'p': 1,
             'r': 1,
             'k': 1})

In [29]:
from functools import reduce
KVcharacters.groupByKey().map(lambda row: (row[0], reduce(addFunc, row[1]))).collect()

[('d', 2),
 ('l', 1),
 ('o', 1),
 ('p', 1),
 ('m', 2),
 ('y', 1),
 ('n', 2),
 ('a', 4),
 ('e', 2),
 ('i', 3),
 ('s', 2),
 ('v', 2),
 ('r', 1),
 ('k', 1)]

In [30]:
KVcharacters.reduceByKey(addFunc).collect()

[('d', 2),
 ('l', 1),
 ('o', 1),
 ('p', 1),
 ('m', 2),
 ('y', 1),
 ('n', 2),
 ('a', 4),
 ('e', 2),
 ('i', 3),
 ('s', 2),
 ('v', 2),
 ('r', 1),
 ('k', 1)]

In [39]:
df = spark.read.option("header", "true").option("inferSchema", "true")\
.csv("data/retail-data/all/")

rdd = df.coalesce(10).rdd
df.printSchema()

root
 |-- PK     �SX�-X��  �     Index/Document.iwa � �_�e8\b4\b �\v"\v: string (nullable = true)



In [44]:
def partitionFunc(key):
    import random
    if key == 17850 or key == 12583:
      return 0
    else:

      return random.randint(1,2)

keyedRDD = rdd.keyBy(lambda row: row[6])

In [48]:
rdd.take(5)
rdd.map(lambda row: len(row)).distinct().collect()

[1]

In [50]:
## Distributed variables
